In [1]:
# --- Step 1: Install necessary libraries (Run this once) ---
!pip install -q git+https://github.com/huggingface/transformers.git accelerate librosa soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.0/521.0 kB 8.8 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0.dev0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [2]:
import torch
import librosa
import numpy as np
import soundfile as sf
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
from typing import List
import os

# Setup Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# --- Step 2: Helper Function to Load Audio ---
def load_wav(path, target_sr=16000):
    """Loads and resamples audio to the target sampling rate."""
    # librosa.load resamples automatically if sr is provided
    wav, sr = librosa.load(path, sr=target_sr)
    return wav

# --- Step 3: The Classification Class ---
class QwenAudioZeroShot:
    """
    Qwen-Audio zero-shot classification wrapper.
    Uses the Instruct model to generate class labels via prompting.
    """
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2-Audio-7B-Instruct", 
        max_seconds: float = 30.0,
    ):
        self.name = f"QwenAudioZeroShot({model_name})"
        print(f"Loading model: {model_name}...")
        
        # Load Processor and Model
        self.processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
        self.model = Qwen2AudioForConditionalGeneration.from_pretrained(
            model_name,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        ).eval()
        
        self.max_seconds = max_seconds
        
    def predict(self, wav_paths: List[str]):
        """
        Generates text predictions for a list of audio paths.
        Returns the raw text response from the model (e.g., "1", "Dysarthria").
        """
        results = []
        
        for p in wav_paths:
            if not os.path.exists(p):
                print(f"Warning: File not found: {p}")
                results.append("Error: File not found")
                continue

            try:
                # 1. Prepare Audio
                # Qwen2-Audio processor handles resampling, but explicit loading is safer for duration checks
                wav = load_wav(p, target_sr=self.processor.feature_extractor.sampling_rate)
                
                # Optional: Trim if too long to avoid OOM
                if len(wav) / 16000 > self.max_seconds:
                    wav = wav[:int(self.max_seconds * 16000)]

                # 2. Construct the Conversation Prompt
                conversation = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "audio", "audio_url": p}, 
                            {"type": "text", "text": (
                                "Listen to this audio specifically for medical speech analysis. "
                                "Classify the speaker into one of these three categories:\n"
                                "1. Healthy\n"
                                "2. Dysarthria\n"
                                "3. Non-Dysarthria\n\n"
                                "Reply ONLY with the number (1, 2, or 3) corresponding to the category."
                            )}
                        ]
                    }
                ]

                # 3. Apply Chat Template & Process Inputs
                text = self.processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
                
                inputs = self.processor(
                    text=[text], 
                    audios=[wav], 
                    return_tensors="pt", 
                    padding=True,
                    sampling_rate=self.processor.feature_extractor.sampling_rate
                ).to(self.model.device)

                # 4. Generate Response
                with torch.no_grad():
                    generated_ids = self.model.generate(**inputs, max_new_tokens=10)
                    
                    # Trim the input tokens to get only the new generated text
                    generated_ids = generated_ids[:, inputs.input_ids.size(1):]
                    response = self.processor.batch_decode(
                        generated_ids, 
                        skip_special_tokens=True, 
                        clean_up_tokenization_spaces=False
                    )[0]
                    
                results.append(response.strip())
                print(f"Processed {p}: {response.strip()}")

            except Exception as e:
                print(f"Error processing {p}: {e}")
                results.append(f"Error: {str(e)}")
            
        return results

# --- Step 4: Execution ---
if __name__ == "__main__":
    # Initialize the model
    classifier = QwenAudioZeroShot()
    
    # ---------------------------------------------------------
    # REPLACE THESE PATHS WITH YOUR REAL PATIENT AUDIO FILES
    # Example: audio_files = ["/kaggle/input/als-dataset/patient_45.wav", "/kaggle/input/als-dataset/control_02.wav"]
    # ---------------------------------------------------------
    
    audio_files = [
        "/kaggle/input/voc-als-all-wav/VOC-ALS_Wav_All/CT001_phonationE.wav",
        "/kaggle/input/voc-als-all-wav/VOC-ALS_Wav_All/ND015_phonationE.wav"
    ]
    
    # Run Inference
    if any("path/to" in f for f in audio_files):
        print("\n[!] Please update the 'audio_files' list with your actual file paths before running.")
    else:
        print(f"\nProcessing {len(audio_files)} files...")
        predictions = classifier.predict(audio_files)
        
        print("\n--- Final Results ---")
        category_map = {"1": "Healthy", "2": "Dysarthria", "3": "Non-Dysarthria"}
        
        for f, pred in zip(audio_files, predictions):
            readable_pred = category_map.get(pred, pred) # Map number to name if possible
            print(f"File: {os.path.basename(f)} | Prediction: {readable_pred}")

Using device: cuda
Loading model: Qwen/Qwen2-Audio-7B-Instruct...


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

Some parameters are on the meta device because they were offloaded to the cpu.



Processing 2 files...


Keyword argument `audios` is not a valid argument for this processor and will be ignored.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processed /kaggle/input/voc-als-all-wav/VOC-ALS_Wav_All/CT001_phonationE.wav: 3
Processed /kaggle/input/voc-als-all-wav/VOC-ALS_Wav_All/ND015_phonationE.wav: 3

--- Final Results ---
File: CT001_phonationE.wav | Prediction: Non-Dysarthria
File: ND015_phonationE.wav | Prediction: Non-Dysarthria
